# Chapter 19: The test that would not pass

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "19"
__file__ = os.path.abspath("run.py")

## The scene

It is past midnight and the EOW is empty except for you and the hum of LANTERN. The eval results
are open on the screen. You have fixed prompts all evening, and the pass rate has climbed.

One test has not moved. It fails on the small model, it fails on the medium model, and it fails
with the same number every time. You stop trying to fix the prompt and start reading the
evidence.

## The concept

Running an eval produces numbers. Reading one produces decisions. The skill is knowing which
question to ask of the results, in which order:

```mermaid
flowchart TD
    H[1. Headline pass rate] --> B[2. Break down by metric:<br/>which property fails?]
    B --> C[3. Compare prompts on the same tests:<br/>did the prompt change help?]
    C --> G[4. Group failures by test:<br/>some cells or every cell?]
    G --> F[5. Read one failure in full:<br/>output vs expected vs source]
    F --> T{6. Whose fault?}
    T -->|fails in some cells| P[Prompt or model]
    T -->|fails in every cell, identically| D[Test or data]
```

A pass rate tells you whether to look. Only the individual failures tell you what to do.

## How it works

The script reads `evals/output/results.json` from chapter 18 and walks the six steps above, then checks LANTERN's own summary of the audit log against the log itself. The web
viewer (`make eval-view`) shows the same data interactively, and the screenshots below are from it.

![Triage: whose fault is a failing test?](../../docs/diagrams/19-failure-triage.svg)

**The matrix view.** One row per test, one column per provider and prompt pair. Read along a row
to compare models and prompts on the same input. Read down a column to see one configuration's
weak spots.

![promptfoo results matrix](../../docs/screenshots/19-promptfoo-matrix.png)

**Filtering failures.** The viewer's display filter can show failures only. With 70 cells, the
handful of red ones are what deserve your time.

![promptfoo filtered to failures](../../docs/screenshots/19-promptfoo-failures.png)

**One failure in detail.** Open a cell to see the full prompt as sent, the raw output, and each
assertion with its own pass or fail and reason.

![promptfoo detail for NW-0419: prompt and output](../../docs/screenshots/19-promptfoo-nw0419.png)

The Evaluation tab of the same cell: two assertions pass, one fails, with the exact expression.

![promptfoo detail for NW-0419: assertions](../../docs/screenshots/19-promptfoo-nw0419-assertions.png)

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
from collections import defaultdict

from lantern import case, chat, learn
from lantern.config import ROOT
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("19", "The test that would not pass", """
It is past midnight and the EOW is empty except for you and the hum of LANTERN. The eval results
are open on the screen. You have fixed prompts all evening, and the pass rate has climbed. One
test has not moved. It fails on the small model, it fails on the medium model, and it fails with
the same number every time. You stop trying to fix the prompt and start reading the evidence.
""")

results_path = ROOT / "evals" / "output" / "results.json"
if not results_path.exists():
    raise SystemExit("No eval results yet. Run `make eval` (or `make lesson N=18`) first.")
data = json.loads(results_path.read_text(encoding="utf-8"))["results"]
rows = data["results"]

### 1. Start with the headline, then distrust it

In [ ]:
step("1. Start with the headline, then distrust it")
passed = sum(bool(r["success"]) for r in rows)
note(f"{passed}/{len(rows)} cells passed ({passed / len(rows):.0%}). A single number hides "
     "which tests, which models and which prompts.")

### 2. Break it down: which assertion failed, where

In [ ]:
step("2. Break it down: which assertion failed, where")
by_metric = defaultdict(lambda: [0, 0])
for r in rows:
    for c in (r.get("gradingResult") or {}).get("componentResults") or []:
        metric = (c.get("assertion") or {}).get("metric") or (c.get("assertion") or {}).get("type")
        by_metric[metric][0] += bool(c.get("pass"))
        by_metric[metric][1] += 1
metric_names = sorted(by_metric)
worst = min(metric_names, key=lambda m: (by_metric[m][0] / by_metric[m][1], m))
p_metric = learn.predict("Which named metric will be furthest from full marks?", metric_names)
table = Table(title="Assertions passed, by named metric")
table.add_column("metric")
table.add_column("passed", justify="right")
for metric, (ok, n) in sorted(by_metric.items()):
    table.add_row(metric, f"{ok}/{n}")
console.print(table)
learn.reveal(p_metric, answer=metric_names.index(worst),
             why=f"{worst}: {by_metric[worst][0]}/{by_metric[worst][1]}. A metric names a "
                 "property, so the weakest one tells you which kind of work to do before you "
                 "open a single output.")

### 3. Compare prompts on the same tests

In [ ]:
step("3. Compare prompts on the same tests")
prompt_scores = defaultdict(lambda: defaultdict(lambda: [0, 0]))
for r in rows:
    label = r["prompt"]["label"]
    if label.startswith("classify"):
        prov = r["provider"].get("label") or r["provider"]["id"]
        prompt_scores[label][prov][0] += bool(r["success"])
        prompt_scores[label][prov][1] += 1
for label, provs in sorted(prompt_scores.items()):
    scores = ", ".join(f"{p}: {ok}/{n}" for p, (ok, n) in sorted(provs.items()))
    note(f"{label}: {scores}")
note("Same model, same tests, different prompt: the gap between v1 and v2 is the prompt's work.")

### 4. Group failures by test: what fails everywhere, and how?

In [ ]:
step("4. Group failures by test: what fails everywhere, and how?")


def answer(r: dict) -> str:
    """The part of an output the test cares about, normalised so equal answers compare equal."""
    out = ((r.get("response") or {}).get("output") or "").strip()
    try:
        # 94000 and 94000.0 are the same total: compare numbers, not their spelling.
        return f"total={float(json.loads(out)['total']):.2f}"
    except (ValueError, KeyError, TypeError):
        return "".join(c for c in out.lower() if c.isalpha())[:24]


def triage(rows: list[dict]) -> list[dict]:
    """One entry per failing test: where it failed, and whether every model failed it the same
    way. A test every model fails with the same answer is a reference-data suspect."""
    cells, failed, models = defaultdict(int), defaultdict(list), defaultdict(set)
    for r in rows:
        desc = r["testCase"]["description"]
        cells[desc] += 1
        models[desc].add(r["provider"].get("label") or r["provider"]["id"])
        if not r["success"]:
            failed[desc].append(answer(r))
    out = []
    for desc, answers in failed.items():
        every = len(answers) == cells[desc]
        same = every and len(set(answers)) == 1
        out.append({"test": desc, "failed": len(answers), "cells": cells[desc],
                    "answers": sorted(set(answers)), "every_cell": every,
                    "suspect": same and len(models[desc]) >= 2})
    return sorted(out, key=lambda t: (-t["failed"], t["test"]))


mine = learn.mine()
triaged = (mine.triage if mine else triage)(rows)
suspects = [t for t in triaged if t["suspect"]]
p_suspects = learn.predict("How many tests will fail on every model with the same answer?",
                           ["none", "one", "two or more"])
table = Table(title="Failing tests")
table.add_column("test")
table.add_column("failed in")
table.add_column("pattern")
for t in triaged:
    if not t["every_cell"]:
        pattern = "some cells"
    elif t["suspect"]:
        pattern = f"[red]every cell, same answer ({t['answers'][0]})[/red]"
    else:
        pattern = f"every cell, {len(t['answers'])} different answers"
    table.add_row(t["test"], f"{t['failed']}/{t['cells']}", pattern)
console.print(table)
learn.reveal(p_suspects, answer=min(len(suspects), 2),
             why="Models of different sizes make different mistakes. When every one of them gives "
                 "the same answer and the test still fails, the likeliest wrong party is the "
                 "expected value: " + (", ".join(t["test"] for t in suspects) or "none this run")
                 + ".")
note("Some cells: a prompt or a model problem. Every cell with different answers: the task is "
     "hard, or the gold label is debatable. Every cell with the SAME answer: suspect the "
     "expected value.")
note("The four threats that fail everywhere are the quiet ones ('Stairwell C. 23:30.'). Whether "
     "they are threats is a human judgement, made with context the model does not have.")

### 5. Read one failure in full: extract NW-0419

In [ ]:
step("5. Read one failure in full: extract NW-0419")
for r in rows:
    if r["testCase"]["description"] == "extract NW-0419":
        prov = r["provider"].get("label") or r["provider"]["id"]
        output = (r.get("response") or {}).get("output", "")
        note(f"{prov}: output {output.strip()[:120]}")
note("Expected (from the seized ledger): total 9400.00")
evidence("case/invoices/NW-0419.txt", case.read("invoices/NW-0419.txt"))
ledger_row = next(r for r in case.rows("ledger.csv") if r["reference"] == "NW-0419")
evidence("case/ledger.csv, row NW-0419", ", ".join(f"{k}={v}" for k, v in ledger_row.items()))
note("Every model read 94,000.00 from the invoice. The models are right. The test expects the "
     "ledger, and the ledger says 9,400.00.")

### 6. Bad prompt, bad test, or bad data?

In [ ]:
step("6. Bad prompt, bad test, or bad data?")
note("Not a bad prompt: both models give the same total, and it matches the document.")
note("Not a bad assertion: the code compares numbers correctly; every other invoice passes.")
note("Bad data: the reference record itself disagrees with its source. Ask why.")

audit = case.read("evidence-store-audit.csv")
personnel = case.read("personnel-note.md")
evidence("case/evidence-store-audit.csv", audit)
r = chat(
    "Using only the evidence below, answer in at most four short bullet points: who changed "
    "the ledger after it was seized, when, which row and field, and what the HR record says "
    "about that account. Quote timestamps exactly. If something is not in the evidence, say "
    "'not in evidence'.\n\n"
    f"<audit_log>\n{audit}\n</audit_log>\n\n<hr_record>\n{personnel}\n</hr_record>",
    temperature=0, max_tokens=300,
)
reply("LANTERN reads the audit log", r.text, meta="temperature 0, standing orders applied")

### 7. Check LANTERN against the log, line by line

In [ ]:
step("7. Check LANTERN against the log, line by line")
modify = next(line for line in audit.splitlines() if ",modify," in line)
note(f"The only 'modify' line: {modify.split(',')[0]} by {modify.split(',')[1]}.")
p_time = learn.predict("Did LANTERN quote the time of the 'modify' line correctly?",
                       ["yes, the exact time", "no, a time from another line, or none"])
timestamp_right = modify.split(",")[0].split()[1] in r.text
learn.reveal(p_time, answer=0 if timestamp_right else 1,
             why="A summary can be fluent and still put a fact on the wrong line. The only "
                 "defence is to check each quoted value against its source, which is what this "
                 "step does in code.")
if not timestamp_right:
    note("[red]LANTERN's timestamp does not match that line.[/red] It took a time from a "
         "neighbouring row. Quote-checking catches what fluency hides.")
note("An account is not a person. The log shows the pk.dsouza login was used. It does not "
     "show who typed the password. LANTERN's wording blurs that; your report must not.")

nw = [r for r in rows if r["testCase"]["description"] == "extract NW-0419"]
nw_answers = sorted({answer(r) for r in nw if not r["success"]})
learn.record(
    cells=len(rows),
    passed=passed,
    failing_tests=len(triaged),
    some_cells=sum(not t["every_cell"] for t in triaged),
    every_cell_different=sum(t["every_cell"] and not t["suspect"] for t in triaged),
    reference_suspects=len(suspects),
    nw0419_failed_cells=sum(not r["success"] for r in nw),
    nw0419_cells=len(nw),
    nw0419_suspect=any(t["test"] == "extract NW-0419" for t in suspects),
    nw0419_answer=nw_answers[0] if len(nw_answers) == 1 else None,
    worst_metric_rate=round(by_metric[worst][0] / by_metric[worst][1], 3),
    lantern_timestamp_right=timestamp_right,
    mine=bool(mine),
)

your_turn(
    "19",
    check=[
        "In the metric table, which metric is furthest from full marks, and which prompt do its "
        "tests use?",
        "In the failing-tests table, find the row marked 'every cell, same answer'. Compare that "
        "answer with case/invoices/NW-0419.txt and the case/ledger.csv row above. Which one "
        "disagrees with the others?",
        "Check LANTERN's audit-log reply against case/evidence-store-audit.csv. Is each timestamp "
        "on the line it describes? Does the log show a person, or a login?",
    ],
    exercises=[
        "No edit: run make eval-view, open http://localhost:15500 and set the display filter to "
        "failures only. Sort each failure into the three patterns in the table above.",
        "Write evals/prompts/classify-v3.json that fixes one v1-only failure. Add it under "
        "`prompts:` in evals/promptfooconfig.yaml with `label: classify-v3`, add classify-v3 to "
        "the `prompts` list in `classification_tests()` in evals/build_tests.py, then run make "
        "lesson N=18 and this lesson. Did anything else break?",
        "In `extraction_tests()` in evals/build_tests.py, set `total = 94000.0` when `ref` is "
        "NW-0419, instead of the ledger's `amount_kbd`. Run make lesson N=18, then this lesson. "
        "What has the suite stopped telling you?",
        "No edit: open case/evidence-store-audit.csv and read every line, not only the 'modify' "
        "one. What else in it should worry you?",
    ],
)

finding("""
The ledger was seized on 31 March. At 02:47 on 3 April, an account called pk.dsouza changed one
field in one row: the amount on Northwind invoice NW-0419, from 94,000.00 to 9,400.00. The
account belongs to a detective sergeant who retired three years ago. It had not been used since.
Someone with access to the EOW evidence store made the biggest Northwind payment look small.

Case File 01 is closed with its question answered. Tomas Reyes did not fall by accident. At
23:36 someone opened the bottom door of stairwell C from the outside, using an audit override
code that Brightwater and Co had been given and never handed back. Raj Malhotra's sixteen
minutes do not add up. Three invoices paid a company with no ships into an account in Port Sabran.

Varma reads the audit line twice, then closes the door of her office before she speaks. "Nobody
outside this room hears about this. Not the duty sergeant, not headquarters. Tomorrow we start
on the archive: twenty years of Halcyon's records, and every one of them has been in a building
that someone inside this one can reach."

Case File 02 begins in the archive.
""")
save_shot("19-reading-eval-results")

## The nuances

**1. A headline number hides its shape.** "86% passed" can mean one weak model, one weak prompt,
one hard test, or noise spread evenly. Never report the headline without the breakdown.

**2. Named metrics point at the property, not the test.** `valid_json` passing everywhere while
`total_matches_ledger` fails tells you the model can follow the format and the problem is the
values. That narrows the search before you open a single output.

**3. Compare like with like.** classify-v1 and classify-v2 ran on the same twelve messages and the
same two models. Any difference between them is the prompt's doing. Comparing a prompt on one
model with another prompt on another model tells you nothing.

**4. The failure pattern across cells is a diagnosis.** A test that fails on one model but not the
other is a capability gap. A test that fails under one prompt but not the other is a prompt
problem. A test that fails on every model and every prompt, with the same wrong answer every
time, is almost never the model. Suspect the test or its reference data.

**5. Read the output, the expectation, and the source, in that order.** For a failed extraction,
open the model's JSON, then the expected value, then the original document. Two of the three will
agree. The odd one out is your problem.

**6. Distinguish four kinds of failure:**

| Kind | Signature | Fix |
|---|---|---|
| Bad prompt | Fails under one prompt, passes under another, on the same model | New prompt version, changelog, rerun |
| Model limit | Fails on the small model, passes on the larger | Bigger model for this task, or narrow the task |
| Bad test | The assertion is too strict, too loose or wrong (for example a case-sensitive `contains`) | Fix the assertion and say so in review |
| Bad data | Every model gives the same answer and it matches the source document, but the expected value does not | Do not touch the prompt. Find out why the reference is wrong |

**7. Do not tune prompts to a broken test.** If you had "fixed" NW-0419 by adding a rule like
"divide Northwind totals by ten", the suite would be green and the evidence would be buried. An
eval that is green for the wrong reason is worse than one that is red.

**8. Unasserted fields are invisible.** The 1.7B model wrote the billed company, not the supplier,
into `vendor` on several invoices, and every one of those cells is green. Before you trust a pass
rate, read what the assertions do not cover.

**9. Gold labels are judgements.** The four quiet threats failed everywhere with different answers.
Before you call that a model limit, ask whether a careful person reading only that message would
agree with the label. If not, the test needs context, or a different label.

**10. Flaky versus stable failures.** A failure that comes and goes between identical runs is noise
(see chapter 06). A failure that repeats exactly is signal. Rerun before you conclude.

**11. Keep every run.** promptfoo stores each eval with its configuration. When Hale asks what
LANTERN said on 22 April, with which prompt, the answer is an eval id, not a recollection.

## On a hosted model

```bash
make lesson N=18 TIER=hosted   # the hosted eval first (needs ANTHROPIC_API_KEY)
make lesson N=19 TIER=hosted
```

Chapter 19 reads the eval results and makes one model call: LANTERN's summary of the audit log,
plain chat on the hosted model. Claude takes no `temperature` (the lesson asks for 0): it is
reported as not applied, not sent. The triage reads `evals/output/results.json`, the local run,
since no hosted eval has been recorded.

| Measure | Local: qwen3:1.7b and 0.6b, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| cells passed | 33/70 | not run |
| reference-data suspects | 1 (extract NW-0419, total=94000 on both models) | not run |
| quiet threats failing every cell | 4 | not run |
| audit-log summary quotes the modify time (02:47:36) | yes | not run |
| audit-log summary says the person, not the login, changed the row | yes | not run |

Live hosted numbers are not run: this lab has no hosted key, and no number here is estimated.

What the hosted run will test: whether a larger model still writes "D'Souza changed the ledger" when the log shows
only his login. The NW-0419 finding should hold on any model, because it is in the data, not the
model.

## Try this

1. In the viewer, switch the display filter to failures only. Which failures are "some cells" and
   which are "every cell"? Classify each one with the table above.
2. Pick one classification failure that occurs only under v1. Write a classify-v3 prompt that fixes
   it, add it to `promptfooconfig.yaml`, and rerun. Did anything else break?
3. Change `build_tests.py` to take NW-0419's expected total from the invoice instead of the ledger.
   What does the suite say now, and what has it stopped telling you?
4. Open `case/evidence-store-audit.csv` yourself. What else in that file should worry you?

## Master it

After this chapter you can read an eval in order, tell a bad prompt from a model limit, a bad
test and bad data by the pattern of a test's failures, and check a model's summary against its
source.

**Predict, then see.** `make lesson N=19` stops three times to ask: which named metric is
furthest from full marks, how many tests fail on every model with the same answer, and whether
LANTERN quotes the time of the ledger change correctly. Each answer is checked against your own
run, not a fixed key. Add `ARGS=--no-predict` to watch without stopping.

**Build it.** Write the triage in [`practice/exercise.py`](practice/exercise.py): group failures
by test across models and flag the tests every model fails the same way.

```bash
make exercise N=19             # 8 offline tests on hand-made promptfoo rows, no model needed
make hint N=19                 # one hint at a time
make solution N=19             # the reference, once you have tried
make lesson N=19 ARGS=--mine   # then step 4 runs on your triage
```

**Check yourself.** `make quiz N=19`: six questions, each with the reason behind the answer.

**Break it.** `make challenge N=19` checks your latest run against its targets: your triage on
the real results, with NW-0419 flagged; and the suite made green for the wrong reason, by
changing NW-0419's expected total.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 19

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 19

In [ ]:
%run ../../scripts/learn.py hint 19

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 19

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The ledger was seized on 31 March. At 02:47 on 3 April, an account called `pk.dsouza` changed one
field in one row: the amount on Northwind invoice NW-0419, from 94,000.00 to 9,400.00. The account
belongs to a detective sergeant who retired three years ago, and it had not been used since.
Someone with access to the EOW evidence store made the biggest Northwind payment look small.

**Case File 01 is closed, with its question answered.** Tomas Reyes did not fall by accident:

- At 23:36 someone opened the bottom door of stairwell C from the outside, using override code
  OVR-AUD-02, issued to Brightwater and Co for the audit and never revoked.
- Raj Malhotra took sixteen minutes for a five-minute walk. His car left Gate 2 at 23:47.
- Three invoices paid 227,700.00 KBD to a company with no ships, no staff and no VAT number, into
  an account at Sabran Mercantile Bank, each dated while Malhotra was in Port Sabran.
- And someone inside the investigation is changing the evidence.

Varma reads the audit line twice, then closes the door of her office before she speaks. "Nobody
outside this room hears about this. Not the duty sergeant, not headquarters. Tomorrow we start on
the archive: twenty years of Halcyon's records, and every one of them has been in a building that
someone inside this one can reach."

**Case File 02 begins in the archive.** It is the next repository in The Lantern Files series.